<center><img src="img/header_gr13_outlier_anomalies.jpg"></center>

# Airbnb Listings
The dataset contains information on **17,432 Airbnb listings in Bangkok**. Each row represents a listing with details such as coordinates, neighborhood, host id, price per night, number of reviews, and so on. 

[Source: InsideAirbnb](http://insideairbnb.com)

## Data Dictionary

| Column                            | Explanation                                                                                                                                                                                        |
| --------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| id                                | Airbnb's unique identifier for the listing                                                                                                                                                         |
| name                              |                                                                                                                                                                                                    |
| host\_id                          |                                                                                                                                                                                                    |
| host\_name                        |                                                                                                                                                                                                    |
| neighbourhood\_group              | The neighbourhood group as geocoded using the latitude and longitude against neighborhoods as defined by open or public digital shapefiles.                                                        |
| neighbourhood                     | The neighbourhood as geocoded using the latitude and longitude against neighborhoods as defined by open or public digital shapefiles.                                                              |
| latitude                          | Uses the World Geodetic System (WGS84) projection for latitude and longitude.                                                                                                                      |
| longitude                         | Uses the World Geodetic System (WGS84) projection for latitude and longitude.                                                                                                                      |
| room\_type                        |                                                                                                                                                                                                    |
| price                             | daily price in local currency. Note, $ sign may be used despite locale                                                                                                                             |
| minimum\_nights                   | minimum number of night stay for the listing (calendar rules may be different)                                                                                                                     |
| number\_of\_reviews               | The number of reviews the listing has                                                                                                                                                              |
| last\_review                      | The date of the last/newest review                                                                                                                                                                 |
| calculated\_host\_listings\_count | The number of listings the host has in the current scrape, in the city/region geography.                                                                                                           |
| availability\_365                 | avaliability\_x. The availability of the listing x days in the future as determined by the calendar. Note a listing may be available because it has been booked by a guest or blocked by the host. |
| number\_of\_reviews\_ltm          | The number of reviews the listing has (in the last 12 months)                                                                                                                                      |
| license                           |                                                                                                                                                                                                    |

The data was compiled by [InsideAirbnb](http://insideairbnb.com) between October and November 2021.

[Source](http://insideairbnb.com/get-the-data.html) and [license](https://creativecommons.org/licenses/by/4.0/) of dataset. 

#     Scenario
You are part of the Trust & Safety Team for a regional travel agency. Your company is launching a "Verified Luxury" program in Bangkok. Before the launch, you must audit the 17,432 listings to ensure that properties claiming to be high-end are legitimate and that no "ghost listings" (fake listings created to scam users) have entered the dataset.

**Background:**
Simple statistical methods like Z-Score or IQR only look at Univariate data (one variable). For example, a price of 20,000 THB might look like an outlier on its own. However, if that listing is a "Private Villa" with 100 reviews and 365 days of availability, it is a perfectly normal luxury listing.

* The problem arises with Multivariate Anomalies. These are listings that look normal in each individual column but are "weird" when you combine them.

* Example: A listing priced at a very low 500 THB (normal) but listed as an "Entire Home" (unusual for that price) with 0 reviews and 0 availability (suspicious).

* Simple filters won't catch this, but Isolation Forest and One-Class SVM will.

**Objective:**
Your objective is to detect Complex Outliers that threaten the integrity of the marketplace. You are trying to catch:

  1. Mislabeled Listings: Properties that claim a high-end room_type but have a price too low to be realistic, or vice versa.

  2. Inactive/Zombie Listings: Listings with high availability_365 but zero number_of_reviews and high minimum_nights. These often represent data entry errors or abandoned accounts that clutter the platform.

  3. The "Hidden" Outlier: Identifying listings that exist in "lonely" parts of the data space—where the combination of price, availability, and guest requirements doesn't match any known successful business model in Bangkok.

<center><img src="img/multivariate.jpg"></center>

# CODE

In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import GridSearchCV

# Load and prepare data
df = pd.read_csv("listings_bangkok.csv")
features = ['price', 'minimum_nights', 'number_of_reviews', 'availability_365']


<center><img src="img/demo/isolation_forest.jpg"></center>

<h1 style="font-weight: bold;"> <center>Isolation Forest</center></h1> 

Isolation Forest is an **unsupervised machine learning algorithm** used for anomaly detection, specifically designed to isolate outliers rather than profile normal data points. By randomly partitioning data, it exploits the fact that anomalies are rare and different, requiring fewer splits to isolate, making it highly efficient, scalable, and effective for high-dimensional datasets. 

Reference: <a href="https://ieeexplore.ieee.org/document/4781136">IEEEXplore</a>

In [ ]:
data1 = df[features].dropna()

model = IsolationForest(random_state=42)

def internal_scorer(estimator, X):
    return np.mean(estimator.score_samples(X))

# Define the "Grid" of parameters we want to test
param_grid = {
    'n_estimators': [100, 200],
    'max_samples': ['auto', 256, 512],
    'contamination': [0.01, 0.05],
    'max_features': [1.0, 0.8]
}
# We use cv=3 to rotate the data 3 times for stability
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring=internal_scorer, cv=3)

#Fit the model
grid_search.fit(data1)

#View the best settings
print("Best parameters found:", grid_search.best_params_)

#Apply the best model
best_model = grid_search.best_estimator_
data1['anomaly'] = best_model.predict(data1)
outliers = data1[data1['anomaly'] == -1]
print(f"Number of outliers detected: {len(outliers)}")
print(outliers.head())



Best parameters found: {'contamination': 0.01, 'max_features': 0.8, 'max_samples': 512, 'n_estimators': 200}
Number of outliers detected: 175
    price  minimum_nights  number_of_reviews  availability_365  anomaly
9    2175               2                208                 0       -1
29    861             358                 56               365       -1
45    700             365                 40               365       -1
48    720             365                 26               364       -1
65    808             120                145               275       -1


<center><img src="img/demo/ocSVM.jpg"></center>

<h1 style="font-weight: bold;"> <center>One-Class SVM</center></h1> 

One-Class SVM is an **unsupervised machine learning algorithm** primarily used for anomaly detection and novelty detection. It trains only on the majority "normal" class to learn a decision boundary, identifying outliers as data points falling outside this boundary. It works efficiently with high-dimensional data, often using RBF kernels to separate data from the origin. 

Reference: <a href="https://www.sciencedirect.com/science/article/abs/pii/S0893608025002953#:~:text=One%2DClass%20Support%20Vector%20Machine%20(OCSVM)%20is%20a%20typical,Manevitz%20&%20Yousef%2C%202001).">ScienceDirect</a>

In [ ]:
data2 = df[features].dropna()

scaler = StandardScaler()
scaled_data = scaler.fit_transform(data2)

def svm_scorer(estimator, X):
    return np.mean(estimator.score_samples(X))

model = OneClassSVM(kernel='rbf')

param_grid = {
    'nu': [0.01, 0.05, 0.1],      # Testing 1%, 5%, and 10% outlier levels
    'gamma': [0.001, 0.01, 0.1]   # Testing how "tight" the fence should be
}

grid_search = GridSearchCV(
    estimator=model, 
    param_grid=param_grid, 
    scoring=svm_scorer, 
    cv=3
)

grid_search.fit(scaled_data)

best_svm = grid_search.best_estimator_
data2['anomaly_svm'] = best_svm.predict(scaled_data)
outliers = data2[data2['anomaly_svm'] == -1]

print(f"Best SVM Settings: {grid_search.best_params_}")
print(f"Total Outliers Found: {len(data2[data2['anomaly_svm'] == -1])}")
print(outliers.head())

Best SVM Settings: {'gamma': 0.001, 'nu': 0.1}
Total Outliers Found: 1743
    price  minimum_nights  number_of_reviews  availability_365  anomaly_svm
9    2175               2                208                 0           -1
14   5431              28                148               362           -1
21   1733               3                205               239           -1
27   1159              30                194               365           -1
29    861             358                 56               365           -1
